In [ ]:
import json
import os
from tqdm.notebook import tqdm
from pydantic import RootModel
import argparse
from typing import List
import ollama
import time
import pandas as pd
import csv
import re
from sklearn.metrics import classification_report


In [ ]:
# HELPER FUNCTIONS

def regex_extract_to_dataframe(strings):
    # Initialize empty lists to store extracted values
    article_ids = []
    about_risingprices_values = []
    relevant_to_ir_values = []

    # Define regex pattern for article_id and about_risingprices with optional double quotes
    article_id_pattern = r'"article_id"\s*:\s*"?(\d+)"?'
    about_risingprices_pattern = r'"main_topic_risingprices"\s*:\s*"?(\d)"?'
    relevant_to_ir_pattern = r'"relevant_to_ir"\s*:\s*"?(\d)"?'

    # Iterate through each string
    for string_data in strings:
        # replace \ with nothing
        string_data = string_data.replace("\\", "")
        string_data = string_data.lower()
        
        # Use regex to find matches for article_id
        article_id_match = re.search(article_id_pattern, string_data)

        # Use regex to find matches for about_risingprices
        about_risingprices_match = re.search(about_risingprices_pattern, string_data)

        # Use regex to find matches for relevant_to_ir
        relevant_to_ir_match = re.search(relevant_to_ir_pattern, string_data)

        # Extract values from the regex matches
        article_id = int(article_id_match.group(1)) if article_id_match else None
        about_risingprices = int(about_risingprices_match.group(1)) if about_risingprices_match else None
        relevant_to_ir = int(relevant_to_ir_match.group(1)) if relevant_to_ir_match else None

        # Append values to the respective lists
        article_ids.append(article_id)
        about_risingprices_values.append(about_risingprices)
        relevant_to_ir_values.append(relevant_to_ir)

    # Create a DataFrame using the extracted values
    df = pd.DataFrame({
        "article_id": article_ids,
        "about_risingprices": about_risingprices_values,
        "relevant_to_ir": relevant_to_ir_values
    })

    return df

def build_prompt(article_id: str, text: str, country:str, country_key:str, system_prompt: str, main_prompt: str):
    """Create prompts in the format expected by Ollama"""
    # Format the prompts with actual values
    # formatted_input = input_prompt.format(article_id=article_id, text=text)
    formatted_main = main_prompt.format(article_id=article_id, text=text, country=country, country_key=country_key)
    
    # Combine user content
    # user_content = formatted_input + formatted_main
    user_content = formatted_main
    
    messages = [
        {
            "role": "system",
            "content": system_prompt.format(country=country)
        },
        {
            "role": "user", 
            "content": user_content
        }
    ]
    return messages

In [ ]:
# ANNOTATED DATA FOR PROMPT TESTING
df = pd.read_csv('cleaned_qualtrics_output_ir_test_v2.csv', encoding='utf-8-sig', sep=';', quoting=csv.QUOTE_NONNUMERIC)
print(df.shape)
df['article_id'] = df['article_id'].astype(int)
# Rename Text to text
df.rename(columns={'Text': 'text'}, inplace=True)
# merge article_title and text for text
df['text'] = df['article_title'] + '\n' + df['text']
df.head()

In [ ]:
# get source from the second digit of article_id
df['source'] = df['article_id'].astype(str).str[1]
df['source'].value_counts()

In [ ]:
df.main_topic.value_counts(dropna=False)

In [ ]:

df['main_topic'] = df['main_topic'].replace("No", 0)
df['main_topic'] = df['main_topic'].replace("Yes", 1)

df['main_topic'] = df['main_topic'].astype(int)

df['relevance'] = df['relevance'].replace("No", 0)
df['relevance'] = df['relevance'].replace("Yes", 1)
df['relevance'] = df['relevance'].astype(int, errors='ignore')


In [ ]:
df.relevance.value_counts(dropna=False)

In [ ]:
system_prompt = """
You are a professional news annotator specializing in identifying whether articles are mainly about increasing cost of living and if so, whether the discussion about this topic in the article is relevant to {country}. 
"""
main_prompt = """
An article is considered mainly about increasing cost of living if the majority of its coverage focuses on this topic. Consider the complete article text. 

Include all forms of discussion related to the increasing cost of living, such as:
-	Inflation indicators: inflation rates, statistics and forecasts, changes in consumer price indices, retail or producer price indices, general consumer price developments, and impact of inflation on increasing cost of living,
-	Causes and contributing factors of rising prices/cost of living: government policies, global economic, political and financial developments, and changes in production or labour markets, inflation,
-	Discussion on interest rates (e.g. mortgages, loans, credit),
-	Impact of increasing prices/cost of living on the public in general, including different income and demographic groups and households,
-	Purchasing power and income: decreasing real wages/real pays due to inflation/increasing costs, income inequality due to rising prices, declining purchasing power, decreasing value of savings,
-	Costs and affordability of essential goods and services such as food, groceries, housing, energy (electricity, heating, gas), oil, fuel, transport, healthcare, childcare, and education,
-	Housing and mortgage costs: costs of rent, mortgage rates, house prices, borrowing costs, interest rates on mortgage loans, property market discussions, affordability of housing,
-	Energy prices: price changes (increasing or decreasing) in oil, fuel, electricity and gas,
-	Poverty and other affordability problems: People struggling to meet living costs, discussions about hunger and poverty, solutions for poverty, policies to reduce poverty or to help people with cost of living pressures,
-	Experiences and reactions from various groups within society (e.g. households, workers, union members, customers) due to rising prices, affordabilty, wages, cost of living pressures in their daily/working lives, including strikes, protests, personal stories,
-	Business and organizational impact: Effects of rising prices and inflation on business costs and operations, strategies to combat increasing costs/inflation, profitability, and workforce decisions, experiences of businesses, industry, and state organizations,
-	Labor relations and pay disputes: pay increase negotiations and wage demands, labour disputes and strikes,
-	Policy discussions/responses: interest rate discussions, central bank policies (incl. European Central Bank for EU countries) for inflation/interest rates, inflation targets, and government measures addressing rising prices or living costs.

# Do **not** count brief mentions of increasing cost of living as main topic if the article is mainly about other issues (e.g. economic situation in the country (growth/recession/crisis), macroeconomic performance and policy (e.g. GDP, state debt), unless the coverage of such issues is directly linked and discussed in relation to increasing cost of living in the larger portion of the article text.

If the article is mainly about increasing cost of living, also decide whether the discussion is relevant to {country} context. When coding this variable, consider only the parts of the article about increasing cost of living and ignore the rest of the article. 

Code this variable as 1 if the article discusses increasing cost of living in {country} or in the world with implications for {country}. Code as 0 if the article discusses increasing cost of living in other countries without direct implications for {country}.

Use the following JSON format for your output. Your output should only contain the JSON object without any additional text or explanation.

```json
{{
  "article_id": "{article_id}",
  "main_topic_risingprices": 1 or 0,
  "relevant_to_{country_key}": 1 or 0
}}
```

Article to annotate:

Article ID: {article_id}
Text: {text}

Task: Using the annotation instructions, decide whether this article is mainly about increasing cost of living. Then, if the article is mainly about increasing cost of living, decide whether the discussion is relevant to {country} context.
- Provide a value (1 or 0) for "relevant_to_{country_key}" **only if main_topic_risingprices = 1**, otherwise set "relevant_to_{country_key}" to 0.
- Output your response in the provided JSON format, without any additional text or explanation.
"""

In [ ]:
# Process dataset
unique_articles = df[['article_id', 'text', 'main_topic']].drop_duplicates(subset=['article_id']).reset_index(drop=True)

In [ ]:
model_name = "llama3:8b" 
ollama.pull(model_name)

In [ ]:
%time
results = []

for idx, row in unique_articles.iterrows():
    article_id = row['article_id']
    text = row['text']
    country = "Ireland"
    country_key = "ir"

    messages = build_prompt(article_id, text, country, country_key, system_prompt, main_prompt)
    
    response = ollama.chat(
        model=model_name,  
        messages=messages,
        options={'temperature': 0.0, 'seed': 0}
    )

    response_content = response.message.content
    print(response_content)

    results.append(response_content)

#1m48s

In [ ]:
coded_df = regex_extract_to_dataframe(results)
coded_df.head()

In [ ]:
# in coded_df if about_risingprices is 0, set relevant_to_ir to None
coded_df.loc[coded_df['about_risingprices'] == 0, 'relevant_to_ir'] = None

In [ ]:
coded_df.about_risingprices.value_counts(dropna=False)

In [ ]:
# left join the coded_df with the original df
df_merged = coded_df.merge(df, on='article_id', how='left')
print(df_merged.shape)

In [ ]:
df_merged.head()
df_merged.main_topic.value_counts(dropna=False)

In [ ]:
pd.crosstab(df_merged['about_risingprices'], df_merged['main_topic'])

In [ ]:
print(classification_report(df_merged['about_risingprices'], df_merged['main_topic']))

In [ ]:
df_merged_risingprices = df_merged[(df_merged['about_risingprices'] == 1) & (df_merged['main_topic'] == 1)]
df_merged_risingprices.shape

In [ ]:
pd.crosstab(df_merged_risingprices['relevance'], df_merged_risingprices['relevant_to_ir'], dropna=False)

In [ ]:
print(classification_report(df_merged_risingprices['relevance'], df_merged_risingprices['relevant_to_ir']))